# 03 — Baseline Models

## Objectives
1. Train traditional ML baselines on hand-crafted color features
2. Evaluate with MAE, RMSE, and R² on stratified test split
3. Establish performance floor before deep learning

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import Ridge
from sklearn.svm import SVR
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import matplotlib.pyplot as plt

RANDOM_STATE = 42
TEST_SIZE = 0.2

print("Baseline model dependencies loaded.")

## 1. Load Features

Load the feature matrix produced by `02_feature_extraction.ipynb`.

In [ ]:
# TODO: Replace with actual feature CSV path once extraction pipeline is complete
FEATURES_CSV = "../data/processed/color_features.csv"

try:
    df = pd.read_csv(FEATURES_CSV)
    target_col = "hb_value"
    feature_cols = [c for c in df.columns if c != target_col and c != "image_path"]
    
    X = df[feature_cols].values
    y = df[target_col].values
    print(f"Features: {X.shape[1]}, Samples: {X.shape[0]}")
except FileNotFoundError:
    print("Feature file not found. Run 02_feature_extraction.ipynb first.")
    X, y = np.array([]), np.array([])

## 2. Train & Evaluate Baselines

In [ ]:
if len(X) > 0:
    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE
    )
    
    scaler = StandardScaler()
    X_train_s = scaler.fit_transform(X_train)
    X_test_s = scaler.transform(X_test)
    
    models = {
        "Ridge Regression": Ridge(alpha=1.0),
        "SVR (RBF)": SVR(kernel="rbf", C=10.0, epsilon=0.1),
        "Gradient Boosting": GradientBoostingRegressor(
            n_estimators=200, max_depth=4, learning_rate=0.05,
            random_state=RANDOM_STATE
        ),
    }
    
    results = []
    for name, model in models.items():
        model.fit(X_train_s, y_train)
        y_pred = model.predict(X_test_s)
        
        mae = mean_absolute_error(y_test, y_pred)
        rmse = np.sqrt(mean_squared_error(y_test, y_pred))
        r2 = r2_score(y_test, y_pred)
        
        results.append({"Model": name, "MAE": mae, "RMSE": rmse, "R²": r2})
        print(f"{name}: MAE={mae:.3f}, RMSE={rmse:.3f}, R²={r2:.3f}")
    
    results_df = pd.DataFrame(results)
    display(results_df)
else:
    print("Skipping — no features loaded.")

## Next Steps

- Use these MAE / R² numbers as the baseline to beat with the ViT model in `model/train.py`
- Target: MAE < 1.0 g/dL, R² > 0.75 (clinical relevance threshold)